# FEATURE SELECTION - VN INDEX

## Import libraries

In [1]:
import os
import sys
import random
from datetime import datetime, timedelta
import pandas as pd
import matplotlib.pylab as plt
import ipynbname
import xgboost as xgb
import seaborn as sns
from tqdm import tqdm
from sklearn.base import clone

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from logger.logger import Logger
from utils.constants import *
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import (
    PostgreSQLConnectionDto,
)
from ta.ta_functions import *

In [2]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Helper functions

In [3]:
def get_weekends(from_date: str, to_date: str):
    start = datetime.strptime(from_date, "%Y-%m-%d")
    end = datetime.strptime(to_date, "%Y-%m-%d")

    weekends = []
    current = start

    while current <= end:
        if current.weekday() in (5, 6):  # 5 = Saturday, 6 = Sunday
            weekends.append(current.strftime("%Y-%m-%d"))
        current += timedelta(days=1)

    return weekends

## Parameters

In [4]:
STOCK_NAME = "vn_index"
NOTEBOOK_NAME = ipynbname.name()
RANDOM_SEED = 18
MAX_TIMESHIFT = 5
MIN_TIMESHIFT = 5
FORECAST_HORIZON = 5
COLUMN_ID = "stock"
TARGET_COLUMN = f"return_{FORECAST_HORIZON}"
DATE_COLUMN = "date"
FEATURE_COLUMNS = []  # all TA indicators

# Inclusive
TRAIN_RANGE = ("2000-01-01", "2021-12-31")
VALIDATION_RANGE = ("2022-01-01", "2023-12-31")
TEST_RANGE = ("2024-01-01", "2026-02-26")

In [5]:
WEEKENDS = get_weekends(TRAIN_RANGE[0], TEST_RANGE[1])
HOLIDAYS = []

DAYOFFS = []
DAYOFFS.extend(WEEKENDS)
DAYOFFS.extend(HOLIDAYS)

DAYOFFS[:10], DAYOFFS[-10:]

(['2000-01-01',
  '2000-01-02',
  '2000-01-08',
  '2000-01-09',
  '2000-01-15',
  '2000-01-16',
  '2000-01-22',
  '2000-01-23',
  '2000-01-29',
  '2000-01-30'],
 ['2026-01-24',
  '2026-01-25',
  '2026-01-31',
  '2026-02-01',
  '2026-02-07',
  '2026-02-08',
  '2026-02-14',
  '2026-02-15',
  '2026-02-21',
  '2026-02-22'])

## Load data

In [6]:
my_logger = Logger(file_name=f"{FEATURE_SELECTION_LOG_FILE_BASE}/vn_index/test")

In [7]:
my_connection_model = PostgreSQLConnectionDto(
    logger=my_logger,
    host=os.getenv("POSTGRES_HOST"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("GOLD_POSTGRES_DATABASE"),
)

In [8]:
my_postgresql_driver = PostgreSQLDriver(logger=my_logger)
my_postgresql_driver.connect(my_connection_model)

<DatabaseExecutionStatus.SUCCESS: 'success'>

In [9]:
vn_index_df = my_postgresql_driver.select(
    schema_name="stock_market", table_name="vn_index"
)

In [10]:
dtype_map = {
    "date": str,
    "open": float,
    "high": float,
    "low": float,
    "close": float,
    "adjust": float,
    "change": float,
    "percent_change": float,
    "matching_volume": float,
    "matching_value": float,
    "negotiate_volume": float,
    "negotiate_value": float,
    "number_of_buy_orders": float,
    "buy_volume": float,
    "average_volume_per_buy_order": float,
    "number_of_sell_orders": float,
    "sell_volume": float,
    "average_volume_per_sell_order": float,
    "net_volume": float,
}

vn_index_df = (
    vn_index_df.astype(dtype_map)
    .dropna(subset=["close"])
    .sort_values(by=["date"])
    .reset_index(drop=True)
)

In [11]:
vn_index_df

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,negotiate_volume,negotiate_value,number_of_buy_orders,buy_volume,average_volume_per_buy_order,number_of_sell_orders,sell_volume,average_volume_per_sell_order,net_volume
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970000,4.740000,8.169220e+06,5.152538e+11,350040.0,2.910304e+10,26816.0,5.268141e+07,1965.000000,3448.0,9.057040e+06,2627.0,4.362437e+07
1,2008-03-08,646.190,646.190,646.190,646.190,646.190,25.363333,4.106667,1.314382e+07,8.298321e+11,769057.0,4.594466e+10,27442.0,5.068024e+07,1852.333333,7486.0,1.712815e+07,2464.0,3.355209e+07
2,2008-03-09,652.240,652.240,652.240,652.240,652.240,21.756667,3.473333,1.811842e+07,1.144410e+12,1188073.0,6.278627e+10,28068.0,4.867907e+07,1739.666667,11523.0,2.519927e+07,2301.0,2.347980e+07
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150000,2.840000,2.309302e+07,1.458989e+12,1607090.0,7.962789e+10,28694.0,4.667790e+07,1627.000000,15561.0,3.327038e+07,2138.0,1.340752e+07
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580000,-2.970000,1.384280e+07,8.374665e+11,226000.0,1.209960e+10,14464.0,1.933131e+07,1337.000000,17910.0,2.811427e+07,1570.0,-8.782960e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6561,2026-02-22,1832.317,1859.236,1829.801,1856.535,1856.535,33.445000,1.837000,7.158068e+08,2.227241e+13,23581058.0,7.924402e+11,476711.0,1.233848e+09,2591.300000,393692.0,1.198445e+09,3047.9,3.540296e+07
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050000,1.980000,7.313611e+08,2.264621e+13,23192200.0,7.774309e+11,486603.0,1.253004e+09,2575.000000,401281.0,1.213944e+09,3025.0,3.906029e+07
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480000,0.400000,9.667547e+08,3.120594e+13,22413147.0,7.795734e+11,598087.0,1.575227e+09,2634.000000,507471.0,1.648273e+09,3248.0,-7.304628e+07
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710000,-0.360000,1.083635e+09,3.567852e+13,62225450.0,1.856894e+12,694266.0,1.862364e+09,2682.000000,597096.0,1.892734e+09,3170.0,-3.036983e+07


## Data transformation

In [12]:
vn_index_df.shape

(6566, 19)

### Remove DAYOFFS

In [13]:
vn_index_df_t1 = vn_index_df[~vn_index_df["date"].isin(DAYOFFS)]
vn_index_df_t1

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,negotiate_volume,negotiate_value,number_of_buy_orders,buy_volume,average_volume_per_buy_order,number_of_sell_orders,sell_volume,average_volume_per_sell_order,net_volume
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,350040.0,2.910304e+10,26816.0,5.268141e+07,1965.0,3448.0,9.057040e+06,2627.0,4.362437e+07
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,1607090.0,7.962789e+10,28694.0,4.667790e+07,1627.0,15561.0,3.327038e+07,2138.0,1.340752e+07
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,226000.0,1.209960e+10,14464.0,1.933131e+07,1337.0,17910.0,2.811427e+07,1570.0,-8.782960e+06
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,290070.0,2.036917e+10,16836.0,2.378011e+07,1412.0,12032.0,2.034621e+07,1691.0,3.433900e+06
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,289160.0,1.702626e+10,13298.0,1.703506e+07,1281.0,12256.0,1.900562e+07,1551.0,-1.970560e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,24358775.0,8.224590e+11,456926.0,1.195538e+09,2623.9,378513.0,1.167449e+09,3093.7,2.808830e+07
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,23192200.0,7.774309e+11,486603.0,1.253004e+09,2575.0,401281.0,1.213944e+09,3025.0,3.906029e+07
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,22413147.0,7.795734e+11,598087.0,1.575227e+09,2634.0,507471.0,1.648273e+09,3248.0,-7.304628e+07
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,62225450.0,1.856894e+12,694266.0,1.862364e+09,2682.0,597096.0,1.892734e+09,3170.0,-3.036983e+07


### Create target

In [14]:
vn_index_df_t2 = vn_index_df_t1.copy()
vn_index_df_t2[f"return_{FORECAST_HORIZON}"] = (
    vn_index_df_t1["close"].shift(-FORECAST_HORIZON) / vn_index_df_t2["close"] - 1
)
vn_index_df_t2

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,negotiate_volume,negotiate_value,number_of_buy_orders,buy_volume,average_volume_per_buy_order,number_of_sell_orders,sell_volume,average_volume_per_sell_order,net_volume,return_5
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,350040.0,2.910304e+10,26816.0,5.268141e+07,1965.0,3448.0,9.057040e+06,2627.0,4.362437e+07,0.005717
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,1607090.0,7.962789e+10,28694.0,4.667790e+07,1627.0,15561.0,3.327038e+07,2138.0,1.340752e+07,-0.064683
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,226000.0,1.209960e+10,14464.0,1.933131e+07,1337.0,17910.0,2.811427e+07,1570.0,-8.782960e+06,-0.078987
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,290070.0,2.036917e+10,16836.0,2.378011e+07,1412.0,12032.0,2.034621e+07,1691.0,3.433900e+06,-0.109411
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,289160.0,1.702626e+10,13298.0,1.703506e+07,1281.0,12256.0,1.900562e+07,1551.0,-1.970560e+06,-0.127826
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,24358775.0,8.224590e+11,456926.0,1.195538e+09,2623.9,378513.0,1.167449e+09,3093.7,2.808830e+07,NaN
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,23192200.0,7.774309e+11,486603.0,1.253004e+09,2575.0,401281.0,1.213944e+09,3025.0,3.906029e+07,NaN
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,22413147.0,7.795734e+11,598087.0,1.575227e+09,2634.0,507471.0,1.648273e+09,3248.0,-7.304628e+07,NaN
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,62225450.0,1.856894e+12,694266.0,1.862364e+09,2682.0,597096.0,1.892734e+09,3170.0,-3.036983e+07,NaN


### Create features

In [15]:
DF_MAP = {}
DF_MAP

{}

#### add_bbands

In [16]:
DF_MAP[add_bbands.__name__] = {}
vn_index_df_add_bbands = add_bbands(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_bbands.__name__]["dataframe"] = vn_index_df_add_bbands
vn_index_df_add_bbands

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_bb_20_bandwidth_slope,close_bb_20_bandwidth_acceleration,close_bb_20_pct_b,close_bb_20_pct_b_slope,close_bb_20_pct_b_gt_1,close_bb_20_pct_b_lt_0,close_bb_20_above_upper,close_bb_20_below_lower,close_bb_20_inside_bands,close_bb_20_position
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-0.003677,0.002547,0.818102,0.053949,False,False,False,False,True,0
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,0.002366,0.006043,0.887543,0.069440,False,False,False,False,True,0
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,0.004535,0.002169,0.905573,0.018031,False,False,False,False,True,0
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,0.003006,-0.001528,0.819302,-0.086272,False,False,False,False,True,0


#### add_dema

In [17]:
DF_MAP[add_dema.__name__] = {}
vn_index_df_add_dema = add_dema(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_dema.__name__]["dataframe"] = vn_index_df_add_dema
vn_index_df_add_dema

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_dema_14_20_bars_since_crossover,close_dema_17_20_dist,close_dema_17_20_dist_abs,close_dema_17_20_dist_pct,close_dema_17_20_dist_slope,close_dema_17_20_dist_acceleration,close_dema_17_20_direction,close_dema_17_20_crossover_up,close_dema_17_20_crossover_dn,close_dema_17_20_bars_since_crossover
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,0,NaN,NaN,NaN,NaN,NaN,1,0,0,0
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,1,NaN,NaN,NaN,NaN,NaN,1,0,0,1
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,2,NaN,NaN,NaN,NaN,NaN,1,0,0,2
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,3,NaN,NaN,NaN,NaN,NaN,1,0,0,3
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,4,NaN,NaN,NaN,NaN,NaN,1,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,1,-0.134640,0.134640,-0.000074,0.833613,-0.171649,-1,0,0,19
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,2,0.755894,0.755894,0.000412,0.890534,0.056921,1,1,0,0
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,3,1.565331,1.565331,0.000850,0.809436,-0.081097,1,0,0,1
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,4,1.886599,1.886599,0.001022,0.321269,-0.488168,1,0,0,2


#### add_ema

In [18]:
DF_MAP[add_ema.__name__] = {}
vn_index_df_add_ema = add_ema(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_ema.__name__]["dataframe"] = vn_index_df_add_ema
vn_index_df_add_ema

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_ema_14_17_direction,close_ema_14_17_dist_slope,close_ema_14_20_dist,close_ema_14_20_dist_abs,close_ema_14_20_direction,close_ema_14_20_dist_slope,close_ema_17_20_dist,close_ema_17_20_dist_abs,close_ema_17_20_direction,close_ema_17_20_dist_slope
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,1,0.566427,2.438652,2.438652,1,0.977212,1.108984,1.108984,1,0.410785
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,1,0.656993,3.585944,3.585944,1,1.147292,1.599283,1.599283,1,0.490298
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,1,0.642918,4.724983,4.724983,1,1.139039,2.095403,2.095403,1,0.496121
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,1,0.307217,5.302515,5.302515,1,0.577532,2.365718,2.365718,1,0.270315


#### add_kama

In [19]:
DF_MAP[add_kama.__name__] = {}
vn_index_df_add_kama = add_kama(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_kama.__name__]["dataframe"] = vn_index_df_add_kama
vn_index_df_add_kama

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_kama_14_17_direction,close_kama_14_17_dist_slope,close_kama_14_20_dist,close_kama_14_20_dist_abs,close_kama_14_20_direction,close_kama_14_20_dist_slope,close_kama_17_20_dist,close_kama_17_20_dist_abs,close_kama_17_20_direction,close_kama_17_20_dist_slope
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-1,1.179498,-21.024512,21.024512,-1,2.200885,-6.515011,6.515011,-1,1.021387
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,-1,1.557808,-18.184390,18.184390,-1,2.840122,-5.232697,5.232697,-1,1.282314
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,-1,6.942553,-10.665243,10.665243,-1,7.519147,-4.656103,4.656103,-1,0.576594
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,-1,4.544684,-5.395817,5.395817,-1,5.269426,-3.931360,3.931360,-1,0.724743


#### add_midpoint

In [20]:
DF_MAP[add_midpoint.__name__] = {}
vn_index_df_add_midpoint = add_midpoint(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_midpoint.__name__]["dataframe"] = vn_index_df_add_midpoint
vn_index_df_add_midpoint

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_midpoint_14_17_direction,close_midpoint_14_17_dist_slope,close_midpoint_14_20_dist,close_midpoint_14_20_dist_abs,close_midpoint_14_20_direction,close_midpoint_14_20_dist_slope,close_midpoint_17_20_dist,close_midpoint_17_20_dist_abs,close_midpoint_17_20_direction,close_midpoint_17_20_dist_slope
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-1,0.0,0.0,0.0,-1,12.535,0.0,0.0,-1,12.535
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,-1,0.0,0.0,0.0,-1,0.000,0.0,0.0,-1,0.000
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,-1,0.0,0.0,0.0,-1,0.000,0.0,0.0,-1,0.000
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,-1,0.0,0.0,0.0,-1,0.000,0.0,0.0,-1,0.000


#### add_midprice

In [21]:
DF_MAP[add_midprice.__name__] = {}
vn_index_df_add_midprice = add_midprice(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_midprice.__name__]["dataframe"] = vn_index_df_add_midprice
vn_index_df_add_midprice

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,midprice_14_17_direction,midprice_14_17_dist_slope,midprice_14_20_dist,midprice_14_20_dist_abs,midprice_14_20_direction,midprice_14_20_dist_slope,midprice_17_20_dist,midprice_17_20_dist_abs,midprice_17_20_direction,midprice_17_20_dist_slope
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-1,0.0,-14.001,14.001,-1,11.637,-14.001,14.001,-1,11.637
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,-1,0.0,0.000,0.000,-1,14.001,0.000,0.000,-1,14.001
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,-1,0.0,0.000,0.000,-1,0.000,0.000,0.000,-1,0.000
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,-1,0.0,0.000,0.000,-1,0.000,0.000,0.000,-1,0.000


#### add_sar

In [22]:
DF_MAP[add_sar.__name__] = {}
vn_index_df_add_sar = add_sar(vn_index_df_t2)
DF_MAP[add_sar.__name__]["dataframe"] = vn_index_df_add_sar
vn_index_df_add_sar

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,sar_004_02_dist,sar_004_02_dist_abs,sar_004_02_dist_pct,sar_004_02_up3,sar_004_02_down3,sar_004_02_trend3,sar_002_02_004_02_dist,sar_002_02_004_02_dist_abs,sar_002_02_004_02_direction,sar_002_02_004_02_dist_slope
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,False,False,0,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,18.150000,18.150000,0.027571,False,False,0,0.000000,0.000000,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,-19.580000,19.580000,-0.030656,False,False,0,0.000000,0.000000,-1,0.000000
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,-14.390000,14.390000,-0.022348,False,False,0,0.000000,0.000000,-1,0.000000
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,-9.906800,9.906800,-0.015298,False,False,0,0.391600,0.391600,1,0.391600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,43.478251,43.478251,0.023510,True,False,1,-29.115426,29.115426,-1,-0.769175
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,45.173001,45.173001,0.024285,True,False,1,-27.775341,27.775341,-1,1.340085
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,46.017000,46.017000,0.024639,True,False,1,-22.261207,22.261207,-1,5.514134
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,30.089600,30.089600,0.016169,False,False,0,-19.175930,19.175930,-1,3.085277


#### add_sma

In [23]:
DF_MAP[add_sma.__name__] = {}
vn_index_df_add_sma = add_sma(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_sma.__name__]["dataframe"] = vn_index_df_add_sma
vn_index_df_add_sma

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_sma_14_17_direction,close_sma_14_17_dist_slope,close_sma_14_20_dist,close_sma_14_20_dist_abs,close_sma_14_20_direction,close_sma_14_20_dist_slope,close_sma_17_20_dist,close_sma_17_20_dist_abs,close_sma_17_20_direction,close_sma_17_20_dist_slope
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-1,0.328634,-4.296750,4.296750,-1,4.132179,-2.551897,2.551897,-1,3.803544
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,-1,0.682101,-1.779179,1.779179,-1,2.517571,-0.716426,0.716426,-1,1.835471
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,1,3.172731,1.806964,1.806964,1,3.586143,-0.303015,0.303015,-1,0.413412
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,1,2.395840,4.503393,4.503393,1,2.696429,-0.002426,0.002426,-1,0.300588


#### add_t3

In [24]:
DF_MAP[add_t3.__name__] = {}
vn_index_df_add_t3 = add_t3(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_t3.__name__]["dataframe"] = vn_index_df_add_t3
vn_index_df_add_t3

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_t3_14_17_direction,close_t3_14_17_dist_slope,close_t3_14_20_dist,close_t3_14_20_dist_abs,close_t3_14_20_direction,close_t3_14_20_dist_slope,close_t3_17_20_dist,close_t3_17_20_dist_abs,close_t3_17_20_direction,close_t3_17_20_dist_slope
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-1,1.836910,-30.298047,30.298047,-1,2.027321,-13.947561,13.947561,-1,0.190411
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,-1,2.165434,-27.594419,27.594419,-1,2.703628,-13.409368,13.409368,-1,0.538193
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,-1,2.409062,-24.339008,24.339008,-1,3.255412,-12.563019,12.563019,-1,0.846349
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,-1,2.517648,-20.738280,20.738280,-1,3.600728,-11.479939,11.479939,-1,1.083080


#### add_tema

In [25]:
DF_MAP[add_tema.__name__] = {}
vn_index_df_add_tema = add_tema(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_tema.__name__]["dataframe"] = vn_index_df_add_tema
vn_index_df_add_tema

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_tema_14_17_direction,close_tema_14_17_dist_slope,close_tema_14_20_dist,close_tema_14_20_dist_abs,close_tema_14_20_direction,close_tema_14_20_dist_slope,close_tema_17_20_dist,close_tema_17_20_dist_abs,close_tema_17_20_direction,close_tema_17_20_dist_slope
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,1,-0.070816,14.808129,14.808129,1,0.499011,5.893655,5.893655,1,0.569827
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,1,0.024605,15.421162,15.421162,1,0.613033,6.482084,6.482084,1,0.588428
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,1,-0.144120,15.696453,15.696453,1,0.275291,6.901495,6.901495,1,0.419411
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,1,-1.015498,14.401121,14.401121,1,-1.295332,6.621661,6.621661,1,-0.279834


#### add_trima

In [26]:
DF_MAP[add_trima.__name__] = {}
vn_index_df_add_trima = add_trima(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_trima.__name__]["dataframe"] = vn_index_df_add_trima
vn_index_df_add_trima

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_trima_14_17_direction,close_trima_14_17_dist_slope,close_trima_14_20_dist,close_trima_14_20_dist_abs,close_trima_14_20_direction,close_trima_14_20_dist_slope,close_trima_17_20_dist,close_trima_17_20_dist_abs,close_trima_17_20_direction,close_trima_17_20_dist_slope
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,1,3.905698,3.910846,3.910846,1,7.258888,-0.923099,0.923099,-1,3.353190
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,1,2.986884,10.158740,10.158740,1,6.247894,2.337911,2.337911,1,3.261011
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,1,2.654353,15.330817,15.330817,1,5.172076,4.855635,4.855635,1,2.517723
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,1,1.772921,19.106508,19.106508,1,3.775692,6.858405,6.858405,1,2.002770


#### add_wma

In [27]:
DF_MAP[add_wma.__name__] = {}
vn_index_df_add_wma = add_wma(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_wma.__name__]["dataframe"] = vn_index_df_add_wma
vn_index_df_add_wma

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_wma_14_17_direction,close_wma_14_17_dist_slope,close_wma_14_20_dist,close_wma_14_20_dist_abs,close_wma_14_20_direction,close_wma_14_20_dist_slope,close_wma_17_20_dist,close_wma_17_20_dist_abs,close_wma_17_20_direction,close_wma_17_20_dist_slope
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,1,1.239975,5.681095,5.681095,1,2.533476,1.816672,1.816672,1,1.293501
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,1,1.375817,8.116500,8.116500,1,2.435405,2.876260,2.876260,1,1.059588
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,1,1.392060,10.469905,10.469905,1,2.353405,3.837605,3.837605,1,0.961345
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,1,0.769486,12.018833,12.018833,1,1.548929,4.617047,4.617047,1,0.779442


#### add_adx

In [28]:
DF_MAP[add_adx.__name__] = {}
vn_index_df_add_adx = add_adx(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_adx.__name__]["dataframe"] = vn_index_df_add_adx
vn_index_df_add_adx

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,adx_20_acceleration,plus_di_20,minus_di_20,plus_di_20_slope,minus_di_20_slope,di_20_distance,di_20_distance_abs,di_20_ratio,trend_20_direction,adx_20_di_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-0.038964,21.661970,21.804955,-0.435387,-1.102040,-0.142985,0.142985,0.993443,-1,2.029571
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,0.222380,22.526901,20.774581,0.864931,-1.030375,1.752321,1.752321,1.084349,1,23.983828
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,0.109831,22.606950,20.154154,0.080049,-0.620427,2.452797,2.452797,1.121702,1,32.596081
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,0.174028,23.240212,19.473285,0.633261,-0.680869,3.766927,3.766927,1.193441,1,49.218052


#### add_aroon

In [29]:
DF_MAP[add_aroon.__name__] = {}
vn_index_df_add_aroon = add_aroon(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_aroon.__name__]["dataframe"] = vn_index_df_add_aroon
vn_index_df_add_aroon

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,aroon_osc_20_slope,aroon_20_distance,aroon_20_distance_abs,aroon_20_ratio,aroon_20_direction,aroon_up_20_gt_70,aroon_down_20_gt_70,aroon_up_20_lt_30,aroon_down_20_lt_30,aroon_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,-1,False,False,False,False,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,-1,False,False,False,False,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,-1,False,False,False,False,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,-1,False,False,False,False,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,-1,False,False,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,5.0,-60.0,60.0,0.000000,-1,False,False,True,False,3600.0
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,5.0,-55.0,55.0,0.000000,-1,False,False,True,False,3025.0
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,105.0,50.0,50.0,2.000000,1,True,False,False,False,2500.0
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,5.0,55.0,55.0,2.222222,1,True,False,False,False,3025.0


#### add_bop

In [30]:
DF_MAP[add_bop.__name__] = {}
vn_index_df_add_bop = add_bop(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_bop.__name__]["dataframe"] = vn_index_df_add_bop
vn_index_df_add_bop

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,bop_hist_17_gt_0,bop_hist_17_lt_0,bop_17_strength,bop_signal_20,bop_signal_20_slope,bop_hist_20,bop_hist_20_slope,bop_hist_20_gt_0,bop_hist_20_lt_0,bop_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,False,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,False,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,False,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,False,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,False,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,True,False,0.346313,0.040075,0.064309,0.635884,0.007591,True,False,0.429832
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,True,False,0.643693,0.120741,0.080666,0.776967,0.141083,True,False,0.697489
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,True,False,0.037542,0.163882,0.043141,0.136836,-0.640131,True,False,0.041149
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,False,True,0.254975,0.177685,0.013803,-0.604127,-0.740963,False,True,0.257625


#### add_cci

In [31]:
DF_MAP[add_cci.__name__] = {}
vn_index_df_add_cci = add_cci(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_cci.__name__]["dataframe"] = vn_index_df_add_cci
vn_index_df_add_cci

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,cci_20_abs,cci_20_direction,cci_20_extreme,cci_20_signal,cci_20_signal_slope,cci_20_hist,cci_20_hist_slope,cci_20_hist_gt_0,cci_20_hist_lt_0,cci_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,-1,0,NaN,NaN,NaN,NaN,False,False,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,-1,0,NaN,NaN,NaN,NaN,False,False,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,-1,0,NaN,NaN,NaN,NaN,False,False,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,-1,0,NaN,NaN,NaN,NaN,False,False,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,-1,0,NaN,NaN,NaN,NaN,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,97.567010,1,0,-51.925631,2.346583,149.492641,21.005170,True,False,14585.549966
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,133.780627,1,1,-45.524479,6.401152,179.305106,29.812465,True,False,23987.549503
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,150.025777,1,1,-36.164868,9.359611,186.190645,6.885539,True,False,27933.396234
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,136.468397,1,1,-25.204324,10.960544,161.672721,-24.517924,True,False,22063.216994


#### add_cmo

In [32]:
DF_MAP[add_cmo.__name__] = {}
vn_index_df_add_cmo = add_cmo(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_cmo.__name__]["dataframe"] = vn_index_df_add_cmo
vn_index_df_add_cmo

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,cmo_20_gt_0,cmo_20_lt_0,cmo_20_extreme,cmo_20_signal,cmo_20_signal_slope,cmo_20_hist,cmo_20_hist_slope,cmo_20_hist_gt_0,cmo_20_hist_lt_0,cmo_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,False,False,0,NaN,NaN,NaN,NaN,False,False,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,False,False,0,NaN,NaN,NaN,NaN,False,False,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,False,False,0,NaN,NaN,NaN,NaN,False,False,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,False,False,0,NaN,NaN,NaN,NaN,False,False,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,False,False,0,NaN,NaN,NaN,NaN,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,True,False,0,3.047313,-0.437304,12.766396,1.583213,True,False,201.884070
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,True,False,0,3.317043,0.269730,15.921497,3.155101,True,False,306.306342
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,True,False,0,3.944006,0.626963,17.617684,1.696187,True,False,379.867047
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,True,False,0,4.874242,0.930236,13.472840,-4.144844,True,False,247.187292


#### add_macd

In [33]:
DF_MAP[add_macd.__name__] = {}
vn_index_df_add_macd = add_macd(vn_index_df_t2)
DF_MAP[add_macd.__name__]["dataframe"] = vn_index_df_add_macd
vn_index_df_add_macd

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,macd_12_26_9_signal_lt_0,macd_12_26_9_hist,macd_12_26_9_hist_slope,macd_12_26_9_hist_acceleration,macd_12_26_9_hist_gt_0,macd_12_26_9_hist_lt_0,macd_12_26_9_hist_abs,macd_12_26_9_cross_above,macd_12_26_9_cross_below,macd_12_26_9_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,False,NaN,NaN,NaN,False,False,NaN,False,False,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,False,NaN,NaN,NaN,False,False,NaN,False,False,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,False,NaN,NaN,NaN,False,False,NaN,False,False,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,False,NaN,NaN,NaN,False,False,NaN,False,False,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,False,NaN,NaN,NaN,False,False,NaN,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,False,5.105726,0.639716,-0.390827,True,False,5.105726,False,False,33.792562
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,False,5.915433,0.809707,0.169991,True,False,5.915433,False,False,52.689508
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,False,6.562926,0.647493,-0.162214,True,False,6.562926,False,False,73.474261
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,False,6.154273,-0.408653,-1.056146,True,False,6.154273,False,False,75.853060


#### add_mfi

In [34]:
DF_MAP[add_mfi.__name__] = {}
vn_index_df_add_mfi = add_mfi(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_mfi.__name__]["dataframe"] = vn_index_df_add_mfi
vn_index_df_add_mfi

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,mfi_20_gt_50,mfi_20_lt_50,mfi_20_extreme,mfi_20_signal,mfi_20_signal_slope,mfi_20_hist,mfi_20_hist_slope,mfi_20_hist_gt_0,mfi_20_hist_lt_0,mfi_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,False,False,0,NaN,NaN,NaN,NaN,False,False,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,False,False,0,NaN,NaN,NaN,NaN,False,False,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,False,False,0,NaN,NaN,NaN,NaN,False,False,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,False,False,0,NaN,NaN,NaN,NaN,False,False,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,False,False,0,NaN,NaN,NaN,NaN,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,False,True,0,51.278531,-1.308505,-1.713156,6.876125,False,True,0.744580
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,True,False,0,50.134781,-1.143750,2.044649,3.757805,True,False,4.456169
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,True,False,0,49.470059,-0.664722,8.577110,6.532461,True,False,69.021456
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,True,False,0,49.329050,-0.141009,15.130772,6.553662,True,False,218.788274


#### add_mom

In [35]:
DF_MAP[add_mom.__name__] = {}
vn_index_df_add_mom = add_mom(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_mom.__name__]["dataframe"] = vn_index_df_add_mom
vn_index_df_add_mom

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,mom_20_pct_slope,mom_20_signal,mom_20_signal_slope,mom_20_hist,mom_20_hist_slope,mom_20_hist_acceleration,mom_20_hist_gt_0,mom_20_hist_lt_0,mom_20_hist_abs,mom_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,0.008184,-34.51205,-8.12275,13.04705,23.66775,8.50825,True,False,13.04705,280.054928
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,0.020380,-38.13505,-3.62300,54.55505,41.50800,17.84025,True,False,54.55505,895.793921
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,0.011373,-39.45905,-1.32400,76.57905,22.02400,-19.48400,True,False,76.57905,2842.614336
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,0.011892,-37.48005,1.97900,95.48005,18.90100,-3.12300,True,False,95.48005,5537.842900


#### add_ppo

In [36]:
DF_MAP[add_ppo.__name__] = {}
vn_index_df_add_ppo = add_ppo(vn_index_df_t2)
DF_MAP[add_ppo.__name__]["dataframe"] = vn_index_df_add_ppo
vn_index_df_add_ppo

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,ppo_12_26_9_signal_lt_0,ppo_12_26_9_hist,ppo_12_26_9_hist_slope,ppo_12_26_9_hist_acceleration,ppo_12_26_9_hist_gt_0,ppo_12_26_9_hist_lt_0,ppo_12_26_9_hist_abs,ppo_12_26_9_cross_above,ppo_12_26_9_cross_below,ppo_12_26_9_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,False,NaN,NaN,NaN,False,False,NaN,False,False,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,False,NaN,NaN,NaN,False,False,NaN,False,False,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,False,NaN,NaN,NaN,False,False,NaN,False,False,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,False,NaN,NaN,NaN,False,False,NaN,False,False,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,False,NaN,NaN,NaN,False,False,NaN,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,True,0.517715,0.012012,-0.049881,True,False,0.517715,False,False,0.188407
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,True,0.516198,-0.001517,-0.013528,True,False,0.516198,False,False,0.252385
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,False,0.506823,-0.009375,-0.007858,True,False,0.506823,False,False,0.310881
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,False,0.441505,-0.065317,-0.055942,True,False,0.441505,False,False,0.297716


#### add_roc

In [37]:
DF_MAP[add_roc.__name__] = {}
vn_index_df_add_roc = add_roc(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_roc.__name__]["dataframe"] = vn_index_df_add_roc
vn_index_df_add_roc

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,roc_20_lt_0,roc_20_signal,roc_20_signal_slope,roc_20_hist,roc_20_hist_slope,roc_20_hist_acceleration,roc_20_hist_gt_0,roc_20_hist_lt_0,roc_20_hist_abs,roc_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,True,-1.791556,-0.464901,0.644180,1.283288,0.451593,True,False,0.644180,0.739117
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,False,-2.000269,-0.208713,2.890860,2.246680,0.963392,True,False,2.890860,2.574573
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,False,-2.078852,-0.078583,4.106714,1.215854,-1.030826,True,False,4.106714,8.327845
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,False,-1.969613,0.109240,5.186634,1.079920,-0.135933,True,False,5.186634,16.685513


#### add_rsi

In [38]:
DF_MAP[add_rsi.__name__] = {}
vn_index_df_add_rsi = add_rsi(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_rsi.__name__]["dataframe"] = vn_index_df_add_rsi
vn_index_df_add_rsi

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,rsi_20_extreme,rsi_20_signal,rsi_20_signal_slope,rsi_20_hist,rsi_20_hist_slope,rsi_20_hist_acceleration,rsi_20_hist_gt_0,rsi_20_hist_lt_0,rsi_20_hist_abs,rsi_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,0,51.523656,-0.218652,6.383198,0.791607,-0.142501,True,False,6.383198,50.471018
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,0,51.658521,0.134865,7.960748,1.577550,0.785944,True,False,7.960748,76.576585
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,0,51.972003,0.313482,8.808842,0.848094,-0.729457,True,False,8.808842,94.966762
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,0,52.437121,0.465118,6.736420,-2.072422,-2.920516,True,False,6.736420,61.796823


#### add_stoch

In [39]:
DF_MAP[add_stoch.__name__] = {}
vn_index_df_add_stoch = add_stoch(vn_index_df_t2)
DF_MAP[add_stoch.__name__]["dataframe"] = vn_index_df_add_stoch
vn_index_df_add_stoch

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,stoch_5_3_3_d_lt_50,stoch_5_3_3_kd_dist,stoch_5_3_3_kd_dist_abs,stoch_5_3_3_kd_direction,stoch_5_3_3_kd_dist_slope,stoch_5_3_3_cross_above,stoch_5_3_3_cross_below,stoch_5_3_3_both_gt_80,stoch_5_3_3_both_lt_20,stoch_5_3_3_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,False,NaN,NaN,-1,NaN,False,False,False,False,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,False,NaN,NaN,-1,NaN,False,False,False,False,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,False,NaN,NaN,-1,NaN,False,False,False,False,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,False,NaN,NaN,-1,NaN,False,False,False,False,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,False,NaN,NaN,-1,NaN,False,False,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,False,-0.911291,0.911291,-1,-0.068000,False,False,True,False,42.225637
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,False,-0.883991,0.883991,-1,0.027300,False,False,True,False,40.220827
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,False,0.438204,0.438204,1,1.322195,True,False,True,False,20.409301
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,False,-4.356554,4.356554,-1,-4.794758,False,True,True,False,172.093467


#### stoch_rsi

In [40]:
DF_MAP[add_stoch_rsi.__name__] = {}
vn_index_df_add_stoch_rsi = add_stoch_rsi(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_stoch_rsi.__name__]["dataframe"] = vn_index_df_add_stoch_rsi
vn_index_df_add_stoch_rsi

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,stoch_rsi_20_5_3_d_lt_50,stoch_rsi_20_5_3_kd_dist,stoch_rsi_20_5_3_kd_dist_abs,stoch_rsi_20_5_3_kd_direction,stoch_rsi_20_5_3_kd_dist_slope,stoch_rsi_20_5_3_cross_above,stoch_rsi_20_5_3_cross_below,stoch_rsi_20_5_3_both_gt_80,stoch_rsi_20_5_3_both_lt_20,stoch_rsi_20_5_3_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,False,NaN,NaN,-1,NaN,False,False,False,False,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,False,NaN,NaN,-1,NaN,False,False,False,False,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,False,NaN,NaN,-1,NaN,False,False,False,False,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,False,NaN,NaN,-1,NaN,False,False,False,False,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,False,NaN,NaN,-1,NaN,False,False,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,False,-7.105427e-14,7.105427e-14,-1,-1.421085e-14,False,False,True,False,3.552714e-12
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,False,-7.105427e-14,7.105427e-14,-1,0.000000e+00,False,False,True,False,3.552714e-12
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,False,-8.526513e-14,8.526513e-14,-1,-1.421085e-14,False,False,True,False,4.263256e-12
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,False,-3.108655e+01,3.108655e+01,-1,-3.108655e+01,False,False,False,False,1.047673e+02


#### add_trix

In [41]:
DF_MAP[add_trix.__name__] = {}
vn_index_df_add_trix = add_trix(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_trix.__name__]["dataframe"] = vn_index_df_add_trix
vn_index_df_add_trix

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,trix_20_lt_0,trix_20_signal,trix_20_signal_slope,trix_20_hist,trix_20_hist_slope,trix_20_hist_acceleration,trix_20_hist_gt_0,trix_20_hist_lt_0,trix_20_hist_abs,trix_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,False,0.170234,-0.011332,-0.107997,0.009206,0.001844,False,True,0.107997,0.006721
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,False,0.158876,-0.011358,-0.097096,0.010901,0.001695,False,True,0.097096,0.005999
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,False,0.147745,-0.011131,-0.084873,0.012223,0.001322,False,True,0.084873,0.005336
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,False,0.137083,-0.010662,-0.072380,0.012493,0.000270,False,True,0.072380,0.004683


#### add_ultosc

In [42]:
DF_MAP[add_ultosc.__name__] = {}
vn_index_df_add_ultosc = add_ultosc(vn_index_df_t2)
DF_MAP[add_ultosc.__name__]["dataframe"] = vn_index_df_add_ultosc
vn_index_df_add_ultosc

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,ultosc_7_14_28_extreme,ultosc_7_14_28_signal,ultosc_7_14_28_signal_slope,ultosc_7_14_28_hist,ultosc_7_14_28_hist_slope,ultosc_7_14_28_hist_acceleration,ultosc_7_14_28_hist_gt_0,ultosc_7_14_28_hist_lt_0,ultosc_7_14_28_hist_abs,ultosc_7_14_28_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,1,66.275105,5.808652,15.061085,-4.842266,-3.982165,True,False,15.061085,471.957036
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,1,71.626025,5.350920,10.702648,-4.358437,0.483828,True,False,10.702648,346.002418
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,1,76.023592,4.397567,7.682686,-3.019962,1.338475,True,False,7.682686,258.954757
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,1,78.799090,2.775498,0.655190,-7.027496,-4.007534,True,False,0.655190,19.298157


#### add_willr

In [43]:
DF_MAP[add_willr.__name__] = {}
vn_index_df_add_willr = add_willr(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_willr.__name__]["dataframe"] = vn_index_df_add_willr
vn_index_df_add_willr

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,willr_20_extreme,willr_20_signal,willr_20_signal_slope,willr_20_hist,willr_20_hist_slope,willr_20_hist_acceleration,willr_20_hist_gt_0,willr_20_hist_lt_0,willr_20_hist_abs,willr_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,0,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,0,-64.625617,0.184906,42.808554,11.727233,6.277068,True,False,42.808554,1206.470804
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,1,-62.653551,1.972066,60.197900,17.389346,5.662113,True,False,60.197900,2862.070022
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,1,-59.878404,2.775147,59.822989,-0.374912,-17.764258,True,False,59.822989,2987.834356
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,1,-56.719179,3.159225,45.504087,-14.318902,-13.943990,True,False,45.504087,1764.871810


#### add_ad

In [44]:
DF_MAP[add_ad.__name__] = {}
vn_index_df_add_ad = add_ad(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_ad.__name__]["dataframe"] = vn_index_df_add_ad
vn_index_df_add_ad

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,ad_17_strength,ad_signal_20,ad_signal_20_slope,ad_hist_20,ad_hist_20_slope,ad_hist_20_acceleration,ad_hist_20_gt_0,ad_hist_20_lt_0,ad_hist_20_abs,ad_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,0.000000e+00,NaN,0.000000e+00,NaN,NaN,False,False,0.000000e+00,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN,False,False,0.000000e+00,0.000000e+00
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,False,False,0.000000e+00,0.000000e+00
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,False,False,0.000000e+00,0.000000e+00
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,False,False,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,1.277202e+18,1.016191e+11,2.408528e+08,2.288101e+09,3.464346e+08,-3.664557e+07,True,False,2.288101e+09,1.343773e+18
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,1.425375e+18,1.018924e+11,2.733178e+08,2.596520e+09,3.084182e+08,-3.801637e+07,True,False,2.596520e+09,1.510489e+18
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,2.907221e+18,1.022311e+11,3.386469e+08,3.217146e+09,6.206260e+08,3.122078e+08,True,False,3.217146e+09,3.086121e+18
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,1.200524e+18,1.024858e+11,2.546906e+08,2.419560e+09,-7.975852e+08,-1.418211e+09,True,False,2.419560e+09,1.313566e+18


#### add_adosc

In [45]:
DF_MAP[add_adosc.__name__] = {}
vn_index_df_add_adosc = add_adosc(vn_index_df_t2)
DF_MAP[add_adosc.__name__]["dataframe"] = vn_index_df_add_adosc
vn_index_df_add_adosc

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,adosc_3_10_signal_slope,adosc_3_10_hist,adosc_3_10_hist_slope,adosc_3_10_hist_acceleration,adosc_3_10_hist_gt_0,adosc_3_10_hist_lt_0,adosc_3_10_hist_abs,adosc_3_10_cross_above,adosc_3_10_cross_below,adosc_3_10_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,False,False,NaN,False,False,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,False,False,NaN,False,False,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,False,False,NaN,False,False,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,False,False,NaN,False,False,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,False,False,NaN,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,2.192859e+08,1.944074e+08,-3.680859e+07,1.856939e+06,True,False,1.944074e+08,False,False,2.358041e+17
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,1.834085e+08,1.604795e+08,-3.392797e+07,2.880622e+06,True,False,1.604795e+08,False,False,2.186402e+17
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,1.914920e+08,2.115056e+08,5.102613e+07,8.495410e+07,True,False,2.115056e+08,False,False,3.394531e+17
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,5.752270e+07,-6.544768e+07,-2.769533e+08,-3.279794e+08,False,True,6.544768e+07,False,False,9.067815e+16


#### add_obv

In [46]:
DF_MAP[add_obv.__name__] = {}
vn_index_df_add_obv = add_obv(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_obv.__name__]["dataframe"] = vn_index_df_add_obv
vn_index_df_add_obv

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,obv_signal_20_slope,obv_hist_20,obv_hist_20_slope,obv_hist_20_acceleration,obv_hist_20_gt_0,obv_hist_20_lt_0,obv_hist_20_abs,obv_20_strength,obv_20_cross_above,obv_20_cross_below
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,0.000000e+00,NaN,NaN,False,False,0.000000e+00,NaN,False,False
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,2.199335e+06,2.089368e+07,2.089368e+07,NaN,True,False,2.089368e+07,4.824983e+14,True,False
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,6.715128e+05,6.379372e+06,-1.451431e+07,-3.540800e+07,True,False,6.379372e+06,8.830837e+13,False,False
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,1.766313e+06,1.677997e+07,1.040060e+07,2.491491e+07,True,False,1.677997e+07,2.041604e+14,False,False
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,2.448577e+06,2.326148e+07,6.481513e+06,-3.919085e+06,True,False,2.326148e+07,2.077271e+14,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,2.112294e+08,2.006679e+09,4.734689e+08,-3.428456e+07,True,False,2.006679e+09,1.373970e+18,False,False
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,2.607657e+08,2.477274e+09,4.705954e+08,-2.873546e+06,True,False,2.477274e+09,1.811782e+18,False,False
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,3.280028e+08,3.116026e+09,6.387519e+08,1.681566e+08,True,False,3.116026e+09,3.012433e+18,False,False
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,1.935611e+08,1.838830e+09,-1.277196e+09,-1.915948e+09,True,False,1.838830e+09,1.992621e+18,False,False


#### add_ht_dcperiod

In [47]:
DF_MAP[add_ht_dcperiod.__name__] = {}
vn_index_df_add_ht_dcperiod = add_ht_dcperiod(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_ht_dcperiod.__name__]["dataframe"] = vn_index_df_add_ht_dcperiod
vn_index_df_add_ht_dcperiod

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,ht_dcperiod_17_strength,ht_dcperiod_signal_20,ht_dcperiod_signal_20_slope,ht_dcperiod_hist_20,ht_dcperiod_hist_20_slope,ht_dcperiod_hist_20_acceleration,ht_dcperiod_hist_20_gt_0,ht_dcperiod_hist_20_lt_0,ht_dcperiod_hist_20_abs,ht_dcperiod_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,11.160998,30.908726,0.852766,8.101281,0.683980,0.140156,True,False,8.101281,12.449614
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,9.612100,31.801006,0.892280,8.476657,0.375376,-0.308604,True,False,8.476657,10.745483
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,0.559722,32.600309,0.799303,7.593380,-0.883277,-1.258653,True,False,7.593380,0.637645
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,4.732601,33.235110,0.634801,6.030611,-1.562769,-0.679492,True,False,6.030611,5.596212


#### add_ht_dcphase

In [48]:
DF_MAP[add_ht_dcphase.__name__] = {}
vn_index_df_add_ht_dcphase = add_ht_dcphase(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_ht_dcphase.__name__]["dataframe"] = vn_index_df_add_ht_dcphase
vn_index_df_add_ht_dcphase

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,ht_dcphase_17_strength,ht_dcphase_signal_20,ht_dcphase_signal_20_slope,ht_dcphase_hist_20,ht_dcphase_hist_20_slope,ht_dcphase_hist_20_acceleration,ht_dcphase_hist_20_gt_0,ht_dcphase_hist_20_lt_0,ht_dcphase_hist_20_abs,ht_dcphase_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,141.330658,277.632141,4.724862,44.886191,-8.475301,-8.014462,True,False,44.886191,168.342927
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,187.880122,282.379101,4.746960,45.096123,0.209932,8.685233,True,False,45.096123,223.536628
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,729.565951,288.140496,5.761395,54.733251,9.637128,9.427196,True,False,54.733251,842.811205
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,85927.432616,261.400233,-26.740263,-254.032502,-308.765753,-318.402881,False,True,254.032502,85229.432776


#### add_ht_phasor

In [49]:
DF_MAP[add_ht_phasor.__name__] = {}
vn_index_df_add_ht_phasor = add_ht_phasor(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_ht_phasor.__name__]["dataframe"] = vn_index_df_add_ht_phasor
vn_index_df_add_ht_phasor

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,ht_phasor_17_strength,ht_phasor_signal_20,ht_phasor_signal_20_slope,ht_phasor_hist_20,ht_phasor_hist_20_slope,ht_phasor_hist_20_acceleration,ht_phasor_hist_20_gt_0,ht_phasor_hist_20_lt_0,ht_phasor_hist_20_abs,ht_phasor_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,39.158526,104.922831,0.575823,5.470315,-97.110844,-27.907566,True,False,5.470315,528.076927
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,58.792541,104.688342,-0.234490,-2.227653,-7.697968,89.412876,False,True,2.227653,17.670764
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,439.569364,106.976203,2.287861,21.734678,23.962331,31.660299,True,False,21.734678,570.539471
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,357.659453,106.178645,-0.797558,-7.576800,-29.311478,-53.273809,False,True,7.576800,228.130145


#### add_ht_sine

In [50]:
DF_MAP[add_ht_sine.__name__] = {}
vn_index_df_add_ht_sine = add_ht_sine(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_ht_sine.__name__]["dataframe"] = vn_index_df_add_ht_sine
vn_index_df_add_ht_sine

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,ht_sine_17_strength,ht_sine_signal_20,ht_sine_signal_20_slope,ht_sine_hist_20,ht_sine_hist_20_slope,ht_sine_hist_20_acceleration,ht_sine_hist_20_gt_0,ht_sine_hist_20_lt_0,ht_sine_hist_20_abs,ht_sine_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,0.003938,-0.661381,0.005566,0.052873,-0.058775,-0.119517,True,False,0.052873,0.002813
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,0.009121,-0.649598,0.011783,0.111934,0.059061,0.117836,True,False,0.111934,0.007930
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,0.080400,-0.615777,0.033821,0.321299,0.209365,0.150304,True,False,0.321299,0.078135
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,0.283061,-0.544919,0.070859,0.673156,0.351857,0.142492,True,False,0.673156,0.284553


#### add_ht_trendmode

In [51]:
DF_MAP[add_ht_trendmode.__name__] = {}
vn_index_df_add_ht_trendmode = add_ht_trendmode(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_ht_trendmode.__name__]["dataframe"] = vn_index_df_add_ht_trendmode
vn_index_df_add_ht_trendmode

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,ht_trendmode_17_strength,ht_trendmode_signal_20,ht_trendmode_signal_20_slope,ht_trendmode_hist_20,ht_trendmode_hist_20_slope,ht_trendmode_hist_20_acceleration,ht_trendmode_hist_20_gt_0,ht_trendmode_hist_20_lt_0,ht_trendmode_hist_20_abs,ht_trendmode_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,0.000000,NaN,0.000000,NaN,NaN,False,False,0.000000,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,False,False,0.000000,0.000000
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,0.000000,0.527641,-0.055541,-0.527641,0.055541,-0.005846,False,True,0.527641,0.000000
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,0.000000,0.477390,-0.050252,-0.477390,0.050252,-0.005290,False,True,0.477390,0.000000
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,0.512320,0.527162,0.049772,0.472838,0.950228,0.899976,True,False,0.472838,0.472838
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,0.433493,0.476956,-0.050206,-0.476956,-0.949794,-1.900022,False,True,0.476956,0.476956


#### add_avgprice

In [52]:
DF_MAP[add_avgprice.__name__] = {}
vn_index_df_add_avgprice = add_avgprice(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_avgprice.__name__]["dataframe"] = vn_index_df_add_avgprice
vn_index_df_add_avgprice

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,avgprice_17_strength,avgprice_signal_20,avgprice_signal_20_slope,avgprice_hist_20,avgprice_hist_20_slope,avgprice_hist_20_acceleration,avgprice_hist_20_gt_0,avgprice_hist_20_lt_0,avgprice_hist_20_abs,avgprice_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,640.140000,NaN,0.000000,NaN,NaN,False,False,0.000000,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,292.820000,641.868571,1.728571,16.421429,16.421429,NaN,True,False,16.421429,298.048929
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,59.987319,641.567755,-0.300816,-2.857755,-19.279184,-35.700612,False,True,2.857755,55.954845
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,9.809314,641.789874,0.222119,2.110126,4.967881,24.247065,True,False,2.110126,10.951556
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,18.385024,642.343219,0.553345,5.256781,3.146655,-1.821227,True,False,5.256781,19.450090
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,66.293131,1817.328776,2.163471,20.552974,1.131779,-0.119135,True,False,20.552974,67.727189
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,263.651063,1820.227702,2.898926,27.539798,6.986824,5.855045,True,False,27.539798,272.251560
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,506.356176,1824.184349,3.956647,37.588151,10.048353,3.061529,True,False,37.588151,526.422052
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,135.057653,1828.126316,3.941967,37.448684,-0.139467,-10.187819,True,False,37.448684,142.398621


#### add_medprice

In [53]:
DF_MAP[add_medprice.__name__] = {}
vn_index_df_add_medprice = add_medprice(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_medprice.__name__]["dataframe"] = vn_index_df_add_medprice
vn_index_df_add_medprice

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,medprice_17_strength,medprice_signal_20,medprice_signal_20_slope,medprice_hist_20,medprice_hist_20_slope,medprice_hist_20_acceleration,medprice_hist_20_gt_0,medprice_hist_20_lt_0,medprice_hist_20_abs,medprice_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,640.140000,NaN,0.000000,NaN,NaN,False,False,0.000000,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,292.820000,641.868571,1.728571,16.421429,16.421429,NaN,True,False,16.421429,298.048929
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,59.987319,641.567755,-0.300816,-2.857755,-19.279184,-35.700612,False,True,2.857755,55.954845
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,9.809314,641.789874,0.222119,2.110126,4.967881,24.247065,True,False,2.110126,10.951556
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,18.385024,642.343219,0.553345,5.256781,3.146655,-1.821227,True,False,5.256781,19.450090
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,81.548376,1815.779578,2.183781,20.745922,1.812719,-0.190813,True,False,20.745922,82.911076
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,345.238435,1818.897238,3.117659,29.617762,8.871841,7.059122,True,False,29.617762,355.102163
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,350.498587,1822.682739,3.785501,35.962261,6.344499,-2.527342,True,False,35.962261,364.297707
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,272.103510,1826.803430,4.120692,39.146570,3.184308,-3.160190,True,False,39.146570,285.965692


#### add_typprice

In [54]:
DF_MAP[add_typprice.__name__] = {}
vn_index_df_add_typprice = add_typprice(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_typprice.__name__]["dataframe"] = vn_index_df_add_typprice
vn_index_df_add_typprice

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,typprice_17_strength,typprice_signal_20,typprice_signal_20_slope,typprice_hist_20,typprice_hist_20_slope,typprice_hist_20_acceleration,typprice_hist_20_gt_0,typprice_hist_20_lt_0,typprice_hist_20_abs,typprice_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,640.140000,NaN,0.000000,NaN,NaN,False,False,0.000000,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,292.820000,641.868571,1.728571,16.421429,16.421429,NaN,True,False,16.421429,298.048929
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,59.987319,641.567755,-0.300816,-2.857755,-19.279184,-35.700612,False,True,2.857755,55.954845
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,9.809314,641.789874,0.222119,2.110126,4.967881,24.247065,True,False,2.110126,10.951556
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,18.385024,642.343219,0.553345,5.256781,3.146655,-1.821227,True,False,5.256781,19.450090
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,87.033454,1817.682490,2.432580,23.109510,1.433420,-0.150886,True,False,23.109510,89.341366
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,351.656858,1820.987967,3.305477,31.402033,8.292523,6.859103,True,False,31.402033,364.200779
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,325.212587,1824.859272,3.871305,36.777395,5.375362,-2.917161,True,False,36.777395,340.068312
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,88.489656,1828.612674,3.753403,35.657326,-1.120069,-6.495431,True,False,35.657326,93.897624


#### add_wclprice

In [55]:
DF_MAP[add_wclprice.__name__] = {}
vn_index_df_add_wclprice = add_wclprice(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_wclprice.__name__]["dataframe"] = vn_index_df_add_wclprice
vn_index_df_add_wclprice

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,wclprice_17_strength,wclprice_signal_20,wclprice_signal_20_slope,wclprice_hist_20,wclprice_hist_20_slope,wclprice_hist_20_acceleration,wclprice_hist_20_gt_0,wclprice_hist_20_lt_0,wclprice_hist_20_abs,wclprice_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,640.140000,NaN,0.000000,NaN,NaN,False,False,0.000000,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,292.820000,641.868571,1.728571,16.421429,16.421429,NaN,True,False,16.421429,298.048929
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,59.987319,641.567755,-0.300816,-2.857755,-19.279184,-35.700612,False,True,2.857755,55.954845
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,9.809314,641.789874,0.222119,2.110126,4.967881,24.247065,True,False,2.110126,10.951556
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,18.385024,642.343219,0.553345,5.256781,3.146655,-1.821227,True,False,5.256781,19.450090
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,89.569712,1818.633946,2.556979,24.291304,1.243771,-0.130923,True,False,24.291304,92.325175
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,354.418171,1822.033332,3.399386,32.294168,8.002864,6.759093,True,False,32.294168,368.226181
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,312.191476,1825.947538,3.914207,37.184962,4.890793,-3.112070,True,False,37.184962,327.413589
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,9.454856,1829.517296,3.569758,33.912704,-3.272258,-8.163052,True,False,33.912704,10.089029


#### add_atr

In [56]:
DF_MAP[add_atr.__name__] = {}
vn_index_df_add_atr = add_atr(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_atr.__name__]["dataframe"] = vn_index_df_add_atr
vn_index_df_add_atr

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,atr_20_normalized,atr_20_signal,atr_20_signal_slope,atr_20_hist,atr_20_hist_slope,atr_20_hist_acceleration,atr_20_hist_gt_0,atr_20_hist_lt_0,atr_20_hist_abs,atr_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,0.016773,31.486679,-0.049345,-0.468781,-0.012386,-0.006237,False,True,0.468781,0.028939
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,0.016627,31.433519,-0.053160,-0.505016,-0.036235,-0.023849,False,True,0.505016,0.045146
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,0.016217,31.324287,-0.109232,-1.037709,-0.532693,-0.496457,False,True,1.037709,0.666131
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,0.016002,31.177045,-0.147242,-1.398796,-0.361087,0.171605,False,True,1.398796,0.711048


#### add_natr

In [57]:
DF_MAP[add_natr.__name__] = {}
vn_index_df_add_natr = add_natr(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_natr.__name__]["dataframe"] = vn_index_df_add_natr
vn_index_df_add_natr

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,natr_20_lt_prev,natr_20_signal,natr_20_signal_slope,natr_20_hist,natr_20_hist_slope,natr_20_hist_acceleration,natr_20_hist_gt_0,natr_20_hist_lt_0,natr_20_hist_abs,natr_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,True,1.729220,-0.005470,-0.051965,-0.001151,-0.000262,False,True,0.051965,0.000344
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,True,1.722884,-0.006335,-0.060187,-0.008222,-0.007071,False,True,0.060187,0.000876
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,True,1.713245,-0.009640,-0.091578,-0.031391,-0.023169,False,True,0.091578,0.003757
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,True,1.702478,-0.010766,-0.102280,-0.010702,0.020688,False,True,0.102280,0.002196


#### add_trange

In [58]:
DF_MAP[add_trange.__name__] = {}
vn_index_df_add_trange = add_trange(vn_index_df_t2, n=list(range(2, 21, 3)))
DF_MAP[add_trange.__name__]["dataframe"] = vn_index_df_add_trange
vn_index_df_add_trange

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,trange_17_strength,trange_signal_20,trange_signal_20_slope,trange_hist_20,trange_hist_20_slope,trange_hist_20_acceleration,trange_hist_20_gt_0,trange_hist_20_lt_0,trange_hist_20_abs,trange_20_strength
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,18.150000,NaN,0.000000,NaN,NaN,False,False,0.000000,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,1.817689,18.286190,0.136190,1.293810,1.293810,NaN,True,False,1.293810,1.850148
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,167.805165,17.038934,-1.247256,-11.848934,-13.142744,-14.436553,False,True,11.848934,170.506164
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,17.418080,15.768560,-1.270375,-12.068560,-0.219625,12.923118,False,True,12.068560,17.982154
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,0.115707,30.549117,-0.074118,-0.704117,-0.130882,0.013777,False,True,0.704117,0.144344
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,0.644751,30.423487,-0.125630,-1.193487,-0.489370,-0.358487,False,True,1.193487,0.733995
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,120.692008,29.248869,-1.174618,-11.158869,-9.965382,-9.476012,False,True,11.158869,124.309805
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,15.886531,28.379453,-0.869416,-8.259453,2.899416,12.864798,False,True,8.259453,16.766690


In [59]:
list(DF_MAP.keys())

['add_bbands',
 'add_dema',
 'add_ema',
 'add_kama',
 'add_midpoint',
 'add_midprice',
 'add_sar',
 'add_sma',
 'add_t3',
 'add_tema',
 'add_trima',
 'add_wma',
 'add_adx',
 'add_aroon',
 'add_bop',
 'add_cci',
 'add_cmo',
 'add_macd',
 'add_mfi',
 'add_mom',
 'add_ppo',
 'add_roc',
 'add_rsi',
 'add_stoch',
 'add_stoch_rsi',
 'add_trix',
 'add_ultosc',
 'add_willr',
 'add_ad',
 'add_adosc',
 'add_obv',
 'add_ht_dcperiod',
 'add_ht_dcphase',
 'add_ht_phasor',
 'add_ht_sine',
 'add_ht_trendmode',
 'add_avgprice',
 'add_medprice',
 'add_typprice',
 'add_wclprice',
 'add_atr',
 'add_natr',
 'add_trange']

## Filter columns

In [60]:
REMOVE_COLUMNS = [
    DATE_COLUMN,
    TARGET_COLUMN,
    "close",
    "open",
    "high",
    "low",
    "adjust",
    "change",
    "percent_change",
    "matching_volume",
    "matching_value",
    "negotiate_volume",
    "negotiate_value",
    "number_of_buy_orders",
    "buy_volume",
    "average_volume_per_buy_order",
    "number_of_sell_orders",
    "sell_volume",
    "average_volume_per_sell_order",
    "net_volume",
]


print(f"Filter feature columns: {REMOVE_COLUMNS}")

for df_name in DF_MAP.keys():
    print(f"\nProcessing dataframe {df_name}")

    DF_MAP[df_name]["feature_columns"] = [
        col for col in DF_MAP[df_name]["dataframe"].columns if col not in REMOVE_COLUMNS
    ]

Filter feature columns: ['date', 'return_5', 'close', 'open', 'high', 'low', 'adjust', 'change', 'percent_change', 'matching_volume', 'matching_value', 'negotiate_volume', 'negotiate_value', 'number_of_buy_orders', 'buy_volume', 'average_volume_per_buy_order', 'number_of_sell_orders', 'sell_volume', 'average_volume_per_sell_order', 'net_volume']

Processing dataframe add_bbands

Processing dataframe add_dema

Processing dataframe add_ema

Processing dataframe add_kama

Processing dataframe add_midpoint

Processing dataframe add_midprice

Processing dataframe add_sar

Processing dataframe add_sma

Processing dataframe add_t3

Processing dataframe add_tema

Processing dataframe add_trima

Processing dataframe add_wma

Processing dataframe add_adx

Processing dataframe add_aroon

Processing dataframe add_bop

Processing dataframe add_cci

Processing dataframe add_cmo

Processing dataframe add_macd

Processing dataframe add_mfi

Processing dataframe add_mom

Processing dataframe add_ppo

P

## Prepare data

In [61]:
print(f"Create X and y dataframes")

for df_name, cfg in DF_MAP.items():
    print(f"\nProcessing dataframe {df_name}")

    df = cfg["dataframe"]
    df = df.dropna()

    # Ensure datetime + sort
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
    df = df.sort_values(by=DATE_COLUMN).reset_index(drop=True)

    cfg["dataframe"] = df

    # Train / Validation split
    train_df = df[df[DATE_COLUMN].between(TRAIN_RANGE[0], TRAIN_RANGE[1])]
    val_df = df[df[DATE_COLUMN].between(VALIDATION_RANGE[0], VALIDATION_RANGE[1])]

    cfg["train_dataframe"] = train_df
    cfg["val_dataframe"] = val_df

    # Features / target
    feature_cols = cfg["feature_columns"]

    cfg["X_train"] = train_df[feature_cols]
    cfg["y_train"] = train_df[TARGET_COLUMN]

    cfg["X_val"] = val_df[feature_cols]
    cfg["y_val"] = val_df[TARGET_COLUMN]

Create X and y dataframes

Processing dataframe add_bbands

Processing dataframe add_dema

Processing dataframe add_ema

Processing dataframe add_kama

Processing dataframe add_midpoint

Processing dataframe add_midprice

Processing dataframe add_sar

Processing dataframe add_sma

Processing dataframe add_t3

Processing dataframe add_tema

Processing dataframe add_trima


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20312\670913343.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20312\670913343.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20312\670913343.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,


Processing dataframe add_wma

Processing dataframe add_adx

Processing dataframe add_aroon

Processing dataframe add_bop

Processing dataframe add_cci

Processing dataframe add_cmo

Processing dataframe add_macd

Processing dataframe add_mfi

Processing dataframe add_mom

Processing dataframe add_ppo

Processing dataframe add_roc

Processing dataframe add_rsi

Processing dataframe add_stoch

Processing dataframe add_stoch_rsi

Processing dataframe add_trix

Processing dataframe add_ultosc

Processing dataframe add_willr

Processing dataframe add_ad

Processing dataframe add_adosc


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20312\670913343.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20312\670913343.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20312\670913343.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,


Processing dataframe add_obv

Processing dataframe add_ht_dcperiod

Processing dataframe add_ht_dcphase

Processing dataframe add_ht_phasor

Processing dataframe add_ht_sine

Processing dataframe add_ht_trendmode

Processing dataframe add_avgprice

Processing dataframe add_medprice

Processing dataframe add_typprice

Processing dataframe add_wclprice

Processing dataframe add_atr

Processing dataframe add_natr


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20312\670913343.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20312\670913343.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20312\670913343.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,


Processing dataframe add_trange


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20312\670913343.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])


## Model

In [62]:
# Model hyperparameters
MODEL_N_ESTIMATORS = 5000
MODEL_MAX_DEPTH = 20
MODEL_LEARNING_RATE = 0.01
MODEL_SUBSAMPLE = 0.6
MODEL_COLSAMPLE_BYTREE = 0.5
MODEL_MIN_CHILD_WEIGHT = 30
MODEL_REG_ALPHA = 1.0
MODEL_REG_LAMBDA = 10.0

# Training / system parameters
MODEL_TREE_METHOD = "hist"
MODEL_DEVICE = "cuda"
MODEL_EARLY_STOPPING_ROUNDS = 100
MODEL_ENABLE_CATEGORICAL = True
MODEL_RANDOM_STATE = RANDOM_SEED

N_REPEATS = 20
TOP_N = 50
CORR_THRESHOLD = 0.95

In [63]:
model = xgb.XGBRegressor(
    n_estimators=MODEL_N_ESTIMATORS,
    max_depth=MODEL_MAX_DEPTH,
    learning_rate=MODEL_LEARNING_RATE,
    subsample=MODEL_SUBSAMPLE,
    colsample_bytree=MODEL_COLSAMPLE_BYTREE,
    min_child_weight=MODEL_MIN_CHILD_WEIGHT,
    reg_alpha=MODEL_REG_ALPHA,
    reg_lambda=MODEL_REG_LAMBDA,
    tree_method=MODEL_TREE_METHOD,
    device=MODEL_DEVICE,
    early_stopping_rounds=MODEL_EARLY_STOPPING_ROUNDS,
    enable_categorical=MODEL_ENABLE_CATEGORICAL,
    random_state=MODEL_RANDOM_STATE,
)

model

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.5
,device,'cuda'
,early_stopping_rounds,100
,enable_categorical,True
,eval_metric,None


## Feature Importance + Feature Selection + Save Data

In [64]:
FEATURE_SELECTION_RESULT_DIR

'../../src/feature_selection/feature_selection_result'

In [65]:
os.makedirs(f"{FEATURE_SELECTION_RESULT_DIR}/{STOCK_NAME}", exist_ok=True)

In [66]:
# for df_name in DF_MAP.keys():
#     print(f"\n{'═' * 60}")
#     print(f"  DATAFRAME: {df_name}")
#     print(f"{'═' * 60}")

#     out_dir = os.path.join(f"{FEATURE_SELECTION_RESULT_DIR}/{STOCK_NAME}", df_name)
#     os.makedirs(out_dir, exist_ok=True)

#     cfg = DF_MAP[df_name]
#     df = cfg["dataframe"]

#     model_copy = clone(model)
#     model_copy.fit(
#         cfg["X_train"],
#         cfg["y_train"],
#         eval_set=[(cfg["X_val"], cfg["y_val"])],
#         verbose=100,
#     )

#     X = df[cfg["feature_columns"]]
#     y = df[TARGET_COLUMN]

#     # ── BASELINE ────────────────────────────────────────────────
#     print("\n[1/7] Baseline prediction (GPU)...")
#     baseline_pred = model_copy.predict(X)
#     baseline_mse = np.mean((y - baseline_pred) ** 2)
#     print(f"      ✔ Baseline MSE : {baseline_mse:.6f}")

#     # ── PERMUTATION IMPORTANCE ──────────────────────────────────
#     print("\n[2/7] Permutation importance (GPU)...")
#     X_vals = X.values
#     y_vals = y.values
#     importances = []

#     for i, col in enumerate(tqdm(X.columns, desc="Features", ncols=70)):
#         col_backup = X_vals[:, i].copy()

#         # Stack N_REPEATS shuffled copies → one big batch
#         X_tiled = np.tile(X_vals, (N_REPEATS, 1))
#         for r in range(N_REPEATS):
#             start = r * len(X_vals)
#             X_tiled[start : start + len(X_vals), i] = np.random.permutation(col_backup)

#         preds = model_copy.predict(X_tiled).reshape(N_REPEATS, -1)
#         y_tiled = np.tile(y_vals, N_REPEATS).reshape(N_REPEATS, -1)
#         scores = np.mean((y_tiled - preds) ** 2, axis=1)

#         X_vals[:, i] = col_backup
#         importances.append(scores.mean() - baseline_mse)

#     # ── IMPORTANCE DATAFRAME ────────────────────────────────────
#     print("\n[3/7] Building importance dataframe...")
#     importance_df = (
#         pd.DataFrame({"feature": X.columns, "importance": importances})
#         .sort_values("importance", ascending=False)
#         .reset_index(drop=True)
#     )
#     importance_df["importance_pct"] = (
#         importance_df["importance"] / importance_df["importance"].sum() * 100
#     )
#     importance_df.to_csv(
#         os.path.join(out_dir, f"{df_name}_importance_before.csv"), index=False
#     )
#     print(f"      ✔ Saved → {df_name}_importance_before.csv")

#     # ── HELPERS ─────────────────────────────────────────────────
#     def plot_top_n(imp_df, title, filename):
#         top = (
#             imp_df.sort_values("importance_pct", ascending=False)
#             .head(TOP_N)
#             .set_index("feature")["importance_pct"]
#             .sort_values(ascending=True)
#         )
#         ax = top.plot(kind="barh", figsize=(12, 12))
#         plt.title(title)
#         plt.ylabel("Importance Percentage")
#         ax.xaxis.set_ticks_position("both")
#         ax.tick_params(axis="x", which="both", top=True, bottom=True, labeltop=True)
#         plt.tight_layout()
#         plt.savefig(os.path.join(out_dir, filename), dpi=400, bbox_inches="tight")
#         plt.close()

#     def plot_corr(features, title, filename):
#         corr = df[features].corr()
#         plt.figure(figsize=(14, 12))
#         sns.heatmap(corr, cmap="coolwarm", center=0, square=True, linewidths=0.5)
#         plt.title(title)
#         plt.tight_layout()
#         plt.savefig(os.path.join(out_dir, filename), dpi=400, bbox_inches="tight")
#         plt.close()

#     def make_title(df_name, text):
#         return f"[{STOCK_NAME}] [{TA_NAME_MAP.get(df_name)}] {text}"

#     # ── PLOTS BEFORE REMOVAL ────────────────────────────────────
#     print("\n[4/7] Plotting top-N importance (before removal)...")
#     plot_top_n(
#         importance_df,
#         make_title(df_name, f"Top {TOP_N} Feature Importance % (Before)"),
#         "feature_importance_plot_before.png",
#     )
#     print(f"      ✔ Saved → feature_importance_plot_before.png")

#     print("\n[5/7] Plotting correlation heatmap (before removal)...")
#     top_features_before = importance_df.head(TOP_N)["feature"].tolist()
#     plot_corr(
#         top_features_before,
#         make_title(df_name, "Feature Correlation Matrix (Before)"),
#         "feature_correlation_plot_before.png",
#     )
#     print(f"      ✔ Saved → feature_correlation_plot_before.png")

#     # ── REMOVE CORRELATED FEATURES ──────────────────────────────
#     print("\n[6/7] Removing highly correlated features...")
#     corr_matrix_abs = df[cfg["feature_columns"]].corr().abs()
#     corr_matrix_abs.to_csv(os.path.join(out_dir, f"{df_name}_corr_matrix_abs.csv"))
#     sorted_features = importance_df["feature"].tolist()

#     selected_features, removed_features = [], set()
#     for feature in sorted_features:
#         if feature in removed_features:
#             continue
#         selected_features.append(feature)
#         correlated = corr_matrix_abs.index[
#             corr_matrix_abs[feature] > CORR_THRESHOLD
#         ].tolist()
#         removed_features.update(f for f in correlated if f != feature)

#     print(f"      ┌─────────────────────────────────")
#     print(f"      │ Threshold  : {CORR_THRESHOLD}")
#     print(f"      │ Total      : {len(sorted_features)}")
#     print(f"      │ Selected   : {len(selected_features)}")
#     print(f"      │ Removed    : {len(removed_features)}")
#     print(f"      └─────────────────────────────────")
#     print(f"      Top kept : {selected_features[:5]}")
#     print(
#         f"      Removed  : {list(removed_features)[:5]}{'...' if len(removed_features) > 5 else ''}"
#     )

#     # ── PLOTS AFTER REMOVAL ─────────────────────────────────────
#     print("\n[7/7] Plotting top-N importance & correlation (after removal)...")
#     importance_df_after = importance_df[
#         importance_df["feature"].isin(selected_features)
#     ].reset_index(drop=True)
#     importance_df_after.to_csv(
#         os.path.join(out_dir, f"{df_name}_importance_after.csv"), index=False
#     )
#     plot_top_n(
#         importance_df_after,
#         make_title(df_name, f"Top {TOP_N} Feature Importance % (After)"),
#         "feature_importance_plot_after.png",
#     )
#     top_features_after = importance_df_after.head(TOP_N)["feature"].tolist()
#     plot_corr(
#         top_features_after,
#         make_title(df_name, "Feature Correlation Matrix (After)"),
#         "feature_correlation_plot_after.png",
#     )
#     print(f"      ✔ Saved → {df_name}_importance_after.csv")
#     print(f"      ✔ Saved → feature_importance_plot_after.png")
#     print(f"      ✔ Saved → feature_correlation_plot_after.png")

#     print(f"\n{'─' * 60}")
#     print(f"  ✅ Done: {df_name}")
#     print(f"{'─' * 60}\n")

## Read Feature Importance Results

In [67]:
TOP_N_IN_EACH_FILE = 50
TOP_N_IN_EACH_FILE

50

In [68]:
FEATURE_SELECTION_RESULT_DIR

'../../src/feature_selection/feature_selection_result'

In [69]:
feature_importance_stock_folder = f"{FEATURE_SELECTION_RESULT_DIR}/{STOCK_NAME}"
feature_importance_stock_folder

'../../src/feature_selection/feature_selection_result/vn_index'

In [70]:
import os
import re

pattern = re.compile(r'.*importance_after\.csv$')

importance_after_list = []

for root, dirs, files in os.walk(feature_importance_stock_folder):
    for file in files:
        if pattern.match(file):
            importance_after_list.append(os.path.join(root, file))

importance_after_list

['../../src/feature_selection/feature_selection_result/vn_index\\add_ad\\add_ad_importance_after.csv',
 '../../src/feature_selection/feature_selection_result/vn_index\\add_adosc\\add_adosc_importance_after.csv',
 '../../src/feature_selection/feature_selection_result/vn_index\\add_adx\\add_adx_importance_after.csv',
 '../../src/feature_selection/feature_selection_result/vn_index\\add_aroon\\add_aroon_importance_after.csv',
 '../../src/feature_selection/feature_selection_result/vn_index\\add_atr\\add_atr_importance_after.csv',
 '../../src/feature_selection/feature_selection_result/vn_index\\add_avgprice\\add_avgprice_importance_after.csv',
 '../../src/feature_selection/feature_selection_result/vn_index\\add_bbands\\add_bbands_importance_after.csv',
 '../../src/feature_selection/feature_selection_result/vn_index\\add_bop\\add_bop_importance_after.csv',
 '../../src/feature_selection/feature_selection_result/vn_index\\add_cci\\add_cci_importance_after.csv',
 '../../src/feature_selection/fea

In [77]:
all_feature_importance_df = pd.concat([pd.read_csv(f).head(TOP_N_IN_EACH_FILE) for f in importance_after_list], ignore_index=True)
all_feature_importance_df = all_feature_importance_df[all_feature_importance_df["importance"] > 0]
all_feature_importance_df = all_feature_importance_df.sort_values("importance", ascending=False)
all_feature_importance_df

,feature,importance,importance_pct
1314,trange_normalized,2.535686e-05,1.975087e+01
578,ht_trendmode_signal_11,2.507467e-05,2.534717e+01
138,atr_20_signal,2.348022e-05,7.360911e+00
1091,close_sma_20,2.129881e-05,1.048457e+01
1332,close_trima_20,2.042496e-05,9.993070e+00
...,...,...,...
489,ht_dcperiod_gt_prev,2.168404e-19,5.295544e-13
512,ht_dcphase_cos,2.168404e-19,1.180671e-11
1593,close_gt_wma_11,1.084202e-19,4.637818e-14
1594,close_wma_2_17_direction,1.084202e-19,4.637818e-14
